# YOLO26s Fine-Tune — Aerial Human Detection (v4)

**Goal:** fine-tune YOLO26s pada dataset Roboflow `mukuntha-rgwdi/aerial-human-detection-lwqjh` v4 (3.479 frame UAV, kelas tunggal `person`, sudah resize 640×640) untuk dipakai onboard UAV.

**Setup dulu (sekali):**
1. Settings → Accelerator = **GPU T4x2**
2. API key Roboflow dibaca dari Kaggle Secret `ROBOFLOW_API_KEY` — jangan menaruh key di notebook.
3. Alternatif tanpa key: upload zip dataset (format YOLOv11) sebagai Kaggle Dataset → zip terdeteksi otomatis di Cell 4.

**Alur:** setup → download dataset → train (MuSGD otomatis, 100 epochs, early stop) → validasi → export ONNX + TensorRT fp16 → kumpulkan artefak.

Estimasi runtime: **60–120 menit** di T4. Kuota Kaggle gratis 30 jam/minggu.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("ROBOFLOW_API_KEY")
except ImportError:
    api_key = None

DATA_DIR = Path("/kaggle/working/aerial-human-detection-4")
INPUT_ZIP = next(Path("/kaggle/input").glob("*/aerial*.zip"), None)
print("API key tersedia:", bool(api_key))


In [ ]:
!pip install -q --upgrade ultralytics roboflow
import ultralytics, roboflow
print("ultralytics", ultralytics.__version__)

In [ ]:
if not DATA_DIR.exists():
    if api_key:
        from roboflow import Roboflow
        rf = Roboflow(api_key=api_key)
        project = rf.workspace("mukuntha-rgwdi").project("aerial-human-detection-lwqjh")
        project.version(4).download("yolov11", location=str(DATA_DIR))
    elif INPUT_ZIP:
        import zipfile
        with zipfile.ZipFile(INPUT_ZIP) as z:
            z.extractall(DATA_DIR)
    else:
        raise SystemExit("Tidak ada API key maupun zip - upload dataset dulu (lihat markdown atas)")
print("dataset siap:", DATA_DIR.exists())

In [ ]:
import yaml
cfg = yaml.safe_load((DATA_DIR / "data.yaml").read_text())
print(cfg["names"])
for split in ["train", "valid", "test"]:
    d = DATA_DIR / split
    n = len(list(d.glob("*.jpg"))) + len(list(d.glob("*.png")))
    print(split, n)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")
model.train(
    data=str(DATA_DIR / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=64,           # total 2 GPU (32/GPU); auto-turun bila OOM
    patience=15,
    cache=True,
    device="0,1",
    project="/kaggle/working/run26s",
    name="train",
    exist_ok=True,
)


In [ ]:
import json
m = model.val(data=str(DATA_DIR / "data.yaml"), device=0)
metrics = {
    "mAP50-95": float(m.box.map),
    "mAP50": float(m.box.map50),
    "precision": float(m.box.mp),
    "recall": float(m.box.mr),
}
print(metrics)
Path("/kaggle/working/metrics.json").write_text(json.dumps(metrics, indent=2))

In [ ]:
best = "/kaggle/working/run26s/train/weights/best.pt"
model = YOLO(best)
model.export(format="onnx", imgsz=640, dynamic=True)
try:
    model.export(format="engine", imgsz=640, half=True)
except Exception as e:
    print("engine export gagal (opsional):", e)

In [ ]:
import shutil
out = Path("/kaggle/working/artifacts")
out.mkdir(exist_ok=True)
for f in ["best.pt", "best.onnx", "best.engine"]:
    src = Path("/kaggle/working/run26s/train/weights") / f
    if src.exists():
        shutil.copy(src, out / f)
for f in out.iterdir():
    print(f.name, f.stat().st_size // 1024, "KB")

## Setelah selesai

1. **Commit & Run** (tombol kanan atas) agar output tersimpan.
2. Buka tab **Output** → download folder `artifacts/` → simpan ke `D:/KRTI/model/`.
3. Verifikasi di laptop:

```bash
.venv\Scripts\python -c "from ultralytics import YOLO; m = YOLO('D:/KRTI/model/best.pt'); r = m.predict('D:/KRTI/test_frame.jpg', conf=0.25, save=True, project='D:/KRTI/out', name='verify26s', exist_ok=True); print(len(r[0].boxes), 'deteksi')"
```

Bandingkan mAP di `metrics.json` dengan baseline yolov8n COCO (37.3).